In [ ]:
import os
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)


# Paths


DATA_PATH = "../dataset/processed"
MODEL_PATH = "../models"
REPORT_PATH = "../reports"

os.makedirs(REPORT_PATH, exist_ok=True)


# Load Test Data (Using First 50%)


print("Loading Test Dataset...")

X_test = pd.read_csv(f"{DATA_PATH}/X_test.csv")
y_test = pd.read_csv(f"{DATA_PATH}/y_test.csv").values.ravel()

print("Original Test Samples :", len(X_test))

# -----------------------------------------------------
# Use only first 50% of test data
# -----------------------------------------------------

half = len(X_test) // 2

X_test = X_test.iloc[:half].reset_index(drop=True)
y_test = y_test[:half]

print("Using Test Samples :", len(X_test))


# Load Model


print("\nLoading XGBoost Model...")

model = joblib.load(f"{MODEL_PATH}/xgboost_model.joblib")

print("Model Loaded Successfully")


# Prediction


THRESHOLD = 0.7

probabilities = model.predict_proba(X_test)[:, 1]

predictions = (probabilities >= THRESHOLD).astype(int)


# Metrics


accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

print("\n====FINAL TEST RESULT ====\n")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix\n")

cm = confusion_matrix(y_test, predictions)

print(cm)

print("\nClassification Report\n")

print(classification_report(y_test, predictions))


# Save Metrics


metrics = {
    "Test Samples": int(len(X_test)),
    "Threshold": THRESHOLD,
    "Accuracy": float(accuracy),
    "Precision": float(precision),
    "Recall": float(recall),
    "F1 Score": float(f1),
    "ROC AUC": float(roc_auc)
}

with open(f"{REPORT_PATH}/final_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)


# Confusion Matrix Plot


disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()

plt.title("Confusion Matrix")

plt.savefig(f"{REPORT_PATH}/confusion_matrix.png")

plt.close()


# ROC Curve


RocCurveDisplay.from_predictions(
    y_test,
    probabilities
)

plt.title("ROC Curve")

plt.savefig(f"{REPORT_PATH}/roc_curve.png")

plt.close()


# Precision Recall Curve


PrecisionRecallDisplay.from_predictions(
    y_test,
    probabilities
)

plt.title("Precision Recall Curve")

plt.savefig(f"{REPORT_PATH}/precision_recall_curve.png")

plt.close()

print("\n========================================")
print("Evaluation Completed Successfully")
print("========================================")

print("\nFiles Saved:")

print("reports/final_metrics.json")
print("reports/confusion_matrix.png")
print("reports/roc_curve.png")
print("reports/precision_recall_curve.png")

Loading Test Dataset...
Original Test Samples : 1272524
Using Test Samples : 636262

Loading XGBoost Model...
Model Loaded Successfully

================ FINAL TEST RESULT ================

Accuracy : 0.9999
Precision: 0.9477
Recall   : 0.9988
F1 Score : 0.9726
ROC AUC  : 0.9998

Confusion Matrix

[[635400     45]
 [     1    816]]

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    635445
           1       0.95      1.00      0.97       817

    accuracy                           1.00    636262
   macro avg       0.97      1.00      0.99    636262
weighted avg       1.00      1.00      1.00    636262


Evaluation Completed Successfully

Files Saved:
reports/final_metrics.json
reports/confusion_matrix.png
reports/roc_curve.png
reports/precision_recall_curve.png
